<a href="https://colab.research.google.com/github/Edubel77/enrutaia/blob/main/notebooks/D5_lab_1_5_keras_texto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Lab 1.5 - Paso 1: cargar los splits limpios desde GitHub y verificar su huella
import hashlib, io, json, time, urllib.request
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import f1_score, accuracy_score

BASE = "https://raw.githubusercontent.com/Edubel77/enrutaia/main/data/limpio/"
MD5_ESPERADO = {
    "train":      "cc0ff7796f8e1d1df56862a9f708d521",
    "validation": "ba68845308293612e717aa7c1a55887c",
}   # el test NO se carga esta semana

datos = {}
for split, esperado in MD5_ESPERADO.items():
    contenido = urllib.request.urlopen(BASE + f"{split}.csv").read()
    huella = hashlib.md5(contenido).hexdigest()
    assert huella == esperado, f"¡{split}.csv cambió! md5 = {huella}"
    datos[split] = pd.read_csv(io.BytesIO(contenido))
    print(f"{split:<10} OK | {len(datos[split]):>5} filas | md5 = {huella}")

train, val = datos["train"], datos["validation"]

# Etiquetas: dominio (texto) -> número entero
dominios = sorted(train["dominio"].unique())
a_id = {d: i for i, d in enumerate(dominios)}
y_train = train["dominio"].map(a_id).to_numpy()
y_val   = val["dominio"].map(a_id).to_numpy()
print(f"\n{len(dominios)} dominios:", dominios)

train      OK | 11158 filas | md5 = cc0ff7796f8e1d1df56862a9f708d521
validation OK |  1915 filas | md5 = ba68845308293612e717aa7c1a55887c

18 dominios: ['alarm', 'audio', 'calendar', 'cooking', 'datetime', 'email', 'general', 'iot', 'lists', 'music', 'news', 'play', 'qa', 'recommendation', 'social', 'takeaway', 'transport', 'weather']


In [2]:
# Lab 1.5 - Paso 2: TextVectorization (vocabulario aprendido SOLO de train)
LARGO = 24          # del EDA: el 99 % de los textos tiene 19 palabras o menos

vectorizador = keras.layers.TextVectorization(
    max_tokens=10_000,              # tamaño máximo del vocabulario
    output_sequence_length=LARGO,   # rellena con 0 o corta a 24 posiciones
)
vectorizador.adapt(train["text"].to_numpy())     # aprende el vocabulario SOLO de train

vocab = vectorizador.get_vocabulary()
print("Tamaño del vocabulario:", len(vocab))
print("Primeras 15 entradas:", vocab[:15])

ejemplo = "Pon música de Shakira"
print(f"\n'{ejemplo}' ->", vectorizador([ejemplo]).numpy()[0])

# Vectorizar los dos splits (una sola vez, fuera del modelo)
X_train = vectorizador(tf.constant(train["text"].tolist())).numpy()
X_val   = vectorizador(tf.constant(val["text"].tolist())).numpy()
print("\nShapes:", X_train.shape, X_val.shape)

desconocidas_val = (X_val == 1).sum() / (X_val > 0).sum()
print(f"Palabras de validación que NO están en el vocabulario: {desconocidas_val:.1%}")

Tamaño del vocabulario: 6589
Primeras 15 entradas: ['', '[UNK]', np.str_('de'), np.str_('la'), np.str_('el'), np.str_('a'), np.str_('en'), np.str_('mi'), np.str_('que'), np.str_('las'), np.str_('por'), np.str_('un'), np.str_('para'), np.str_('es'), np.str_('favor')]

'Pon música de Shakira' -> [  25   51    2 1480    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0]

Shapes: (11158, 24) (1915, 24)
Palabras de validación que NO están en el vocabulario: 4.7%


In [3]:
# Lab 1.5 - Paso 3: E1 - Embedding + promedio + softmax
def nuevo_early_stopping():
    return keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

def crear_e1(semilla):
    tf.keras.utils.set_random_seed(semilla)
    m = keras.Sequential([
        keras.Input(shape=(LARGO,), dtype="int64"),
        keras.layers.Embedding(input_dim=len(vocab), output_dim=64, mask_zero=True),
        keras.layers.GlobalAveragePooling1D(),        # promedio de las palabras (ignora el relleno)
        keras.layers.Dropout(0.3),
        keras.layers.Dense(len(dominios), activation="softmax"),
    ], name="E1")
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

def entrenar_y_medir(semilla):
    m = crear_e1(semilla)
    inicio = time.time()
    h = m.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=50,
              batch_size=32, verbose=0, callbacks=[nuevo_early_stopping()])
    seg = time.time() - inicio
    pred = m.predict(X_val, verbose=0).argmax(axis=1)
    return {"semilla": semilla,
            "macro_f1": f1_score(y_val, pred, average="macro"),
            "accuracy": accuracy_score(y_val, pred),
            "epocas": len(h.history["loss"]),
            "segundos": seg}, m, h

r, modelo_e1, hist_e1 = entrenar_y_medir(42)
modelo_e1.summary()

e0 = json.load(urllib.request.urlopen(
    "https://raw.githubusercontent.com/Edubel77/enrutaia/main/results/E0_tfidf.json"))
print(f"\nE0 (TF-IDF)  -> macro-F1: {e0['macro_f1']:.4f} | exactitud: {e0['accuracy']:.4f}")
print(f"E1 (Keras)   -> macro-F1: {r['macro_f1']:.4f} | exactitud: {r['accuracy']:.4f} | "
      f"épocas: {r['epocas']} | {r['segundos']:.0f} s")

Model: "E1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 24, 64)         │       421,696 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 18)             │         1,170 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,268,600 (4.84 MB)

 Trainable params: 422,866 (1.61 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 845,734 (3.23 MB)


E0 (TF-IDF)  -> macro-F1: 0.8725 | exactitud: 0.8789
E1 (Keras)   -> macro-F1: 0.8715 | exactitud: 0.8799 | épocas: 14 | 38 s


In [4]:
# Lab 1.5 - Paso 4: E1 con 3 semillas -> media ± desviación
semillas = [42, 123, 2024]
corridas = [entrenar_y_medir(s)[0] for s in semillas]

tabla = pd.DataFrame(corridas)
print(tabla.round(4).to_string(index=False))

media, desv = tabla["macro_f1"].mean(), tabla["macro_f1"].std()
print(f"\nE1 macro-F1: {media:.4f} ± {desv:.4f}  (3 semillas)")
print(f"E0 macro-F1: {e0['macro_f1']:.4f}  (determinista)")
print(f"Diferencia E1 - E0: {media - e0['macro_f1']:+.4f}")
print("¿E0 cae dentro de media ± 1 desviación de E1?",
      media - desv <= e0["macro_f1"] <= media + desv)

# Reproducibilidad exacta: ¿la semilla 42 repite el mismo número?
repeticion = entrenar_y_medir(42)[0]["macro_f1"]
print(f"\nSemilla 42, 1.ª vez: {corridas[0]['macro_f1']:.6f} | 2.ª vez: {repeticion:.6f}")

# Guardar el resultado (las métricas salen del código)
resultado = {
    "experimento": "E1",
    "modelo": "TextVectorization(10k, largo 24) + Embedding(64) + GlobalAveragePooling1D + Dropout(0.3) + Dense softmax",
    "datos_md5": MD5_ESPERADO,
    "split_evaluado": "validation",
    "semillas": semillas,
    "macro_f1_por_semilla": [round(c["macro_f1"], 4) for c in corridas],
    "macro_f1_media": round(media, 4),
    "macro_f1_desv": round(desv, 4),
    "accuracy_media": round(tabla["accuracy"].mean(), 4),
    "tensorflow": tf.__version__,
}
import os; os.makedirs("results", exist_ok=True)
with open("results/E1_keras.json", "w") as f:
    json.dump(resultado, f, indent=2, ensure_ascii=False)
print("\nGuardado en results/E1_keras.json")

 semilla  macro_f1  accuracy  epocas  segundos
      42    0.8715    0.8799      14   44.3144
     123    0.8728    0.8804      14   35.4723
    2024    0.8753    0.8825      14   31.0420

E1 macro-F1: 0.8732 ± 0.0019  (3 semillas)
E0 macro-F1: 0.8725  (determinista)
Diferencia E1 - E0: +0.0007
¿E0 cae dentro de media ± 1 desviación de E1? True

Semilla 42, 1.ª vez: 0.871508 | 2.ª vez: 0.871508

Guardado en results/E1_keras.json


In [5]:
# Lab 1.5 - Paso 5: reproducir E0 desde los datos del repo
from sklearn.pipeline import make_pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

e0_repro = make_pipeline(
    FeatureUnion([
        ("palabras",   TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("caracteres", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
    ]),
    LogisticRegression(max_iter=2000),
)
e0_repro.fit(train["text"], train["dominio"])
f1_repro = f1_score(val["dominio"], e0_repro.predict(val["text"]), average="macro")
print(f"E0 guardado (Día 4): {e0['macro_f1']:.4f} | E0 reproducido hoy: {f1_repro:.4f}")
print("¿Coinciden?", round(f1_repro, 4) == e0["macro_f1"])

E0 guardado (Día 4): 0.8725 | E0 reproducido hoy: 0.8725
¿Coinciden? True
